In [1]:
import pandas as pd
import sys
import os

# Agrega la ruta de src al sys.path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
#!pip install xgboost
from modules.data_utils import DataProcessor
from modules.eda_utils import *
from modules.get1_cleaning import SalesCleaner
from modules.get2_engineering import FeatureEngineering
from modules.get3_preparing import DataPreparation
from modules.get4_select_model import ModelTraining

# pd.options.display.float_format = '{:.2f}'.format

In [2]:
data_path = "../data/raw/"
sales_train = pd.read_csv(data_path + "sales_train.csv")
shops = pd.read_csv(data_path + "shops.csv")
items = pd.read_csv(data_path + "items.csv")
item_categories = pd.read_csv(data_path + "item_categories.csv")
test = pd.read_csv(data_path + "test.csv")
# submission = pd.read_csv(data_path + "sample_submission.csv")


In [3]:
sales_train = SalesCleaner(sales_train).execute_transformations()
shops = DataProcessor.translate_column(shops, "shop_name")
item_categories = DataProcessor.translate_column(item_categories, "item_category_name")
train = sales_train.merge(shops, on = "shop_id", how = "left")
train = train.merge(items, on = "item_id", how = "left")
train = train.merge(item_categories, on = "item_category_id", how = "left")
summary_df_ip, filtered_data_ip = detect_outliers(train, "item_price", iqr_factor=1.5, z_threshold=3, lower_percentile=0.01, upper_percentile=0.99)
train = filtered_data_ip["Z-Score"].reset_index(drop=True)
train["date"] = pd.to_datetime(train["date"], format="%Y-%m-%d")
t = train.copy()
train_model = FeatureEngineering(t, target="item_cnt_month").apply_feature_engineering()
data_prep = DataPreparation(train_model, target="item_cnt_month")
X_train, X_val, y_train, y_val = data_prep.prepare_data()
model_trainer = ModelTraining(X_train, X_val, y_train, y_val, "random_forest")
full_data, best_model, train_rmse, val_rmse = model_trainer.train_and_evaluate_model()


# train.to_csv("../data/train.csv", index=False)

/Users/danielmpr/Documents/ITAM/ArquitecturaGranEscala/ArquitecturaGranEscala/src/modules/get3_preparing.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[feature] = df[feature].fillna(0)
/Users/danielmpr/Documents/ITAM/ArquitecturaGranEscala/ArquitecturaGranEscala/src/modules/get3_preparing.py:116: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[columns_to_use] = df[columns_to_use].fillna(df[columns_to_use].median())
/Users/danielmpr/Documents/ITAM/ArquitecturaGranEscala/ArquitecturaGranEscala/src/